In [3]:
try:

    !pip install -r requirements.txt

    print(
        "Dependencies installed successfully!"
    )

except Exception as e:

    print(
        "Installation Error:"
    )

    print(e)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 767.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.0/349.0 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

Dependencies installed successfully!


In [1]:
import os
import pickle
import chromadb
import pandas as pd

from datasets import Dataset

from google import genai
from google.colab import userdata

from groq import Groq

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
import ragas

try:
    print("RAGAS imported successfully!")
    print("Version:", ragas.__version__)

except Exception as e:
    print(e)

RAGAS imported successfully!
Version: 0.4.3


In [3]:
# Initialize Gemini and Groq

try:
    gemini_client = genai.Client(
        api_key=userdata.get(
            "GEMINI_KEY"
        )
    )

    print("Gemini initialized!")

except Exception as e:
    print("Gemini Error:")
    print(e)

try:
    groq_client = Groq(
        api_key=userdata.get(
            "GROQ_KEY"
        )
    )

    print("Groq initialized!")

except Exception as e:
    print("Groq Error:")
    print(e)

Gemini initialized!
Groq initialized!


In [4]:
# Load ChromaDB
try:

    chroma_client = chromadb.PersistentClient(
        path="./networking_chromadb"
    )

    collection = (
        chroma_client.get_or_create_collection(
            name="networking_docs"
        )
    )

    print(
        "Records:",
        collection.count()
    )

except Exception as e:

    print(e)

Records: 0


In [5]:
try:

    with open(
        "chunks.pkl",
        "rb"
    ) as f:

        chunks = pickle.load(f)

    with open(
        "chunk_embeddings.pkl",
        "rb"
    ) as f:

        chunk_embeddings = pickle.load(f)

    if collection.count() == 0:

        collection.add(
            ids=[
                f"chunk_{i}"
                for i in range(len(chunks))
            ],
            documents=chunks,
            embeddings=chunk_embeddings
        )

    print(
        "Collection ready!"
    )

    print(
        "Records:",
        collection.count()
    )

except Exception as e:

    print(e)

Collection ready!
Records: 247


In [8]:
try:

    def load_evaluation_data(
        path="arrays.txt"
    ):

        globals_dict = {}

        with open(
            path,
            "r"
        ) as f:

            exec(
                f.read(),
                globals_dict
            )

        return (
            globals_dict["test_questions"],
            globals_dict["ground_truths"]
        )

    test_questions, ground_truths = (
        load_evaluation_data()
    )

    # Keep only first 5 questions
    test_questions = test_questions[:10]
    ground_truths = ground_truths[:10]

    print(
        "Questions:",
        len(test_questions)
    )

    print(
        "Ground Truths:",
        len(ground_truths)
    )

except Exception as e:

    print(
        "Loading Error:"
    )

    print(e)

Questions: 10
Ground Truths: 10


In [9]:
# RAG Query Function
#Generate answers by retrieving relevant chunks from ChromaDB and using Groq for response generation.
try:

    def ask_question(
        question
    ):

        query_response = (
            gemini_client.models.embed_content(
                model="models/gemini-embedding-001",
                contents=question
            )
        )

        query_embedding = (
            query_response.embeddings[0].values
        )

        results = collection.query(
            query_embeddings=[
                query_embedding
            ],
            n_results=5
        )

        retrieved_chunks = (
            results["documents"][0]
        )

        context = "\n\n".join(
            retrieved_chunks
        )

        prompt = f"""
You are an assistant.

Answer only using the provided context.

Context:
{context}

Question:
{question}
"""

        response = (
            groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )
        )

        return {
            "answer":
            response.choices[0]
            .message.content,

            "contexts":
            retrieved_chunks
        }

    print(
        "Function created successfully!"
    )

except Exception as e:

    print(e)

Function created successfully!


In [10]:
# Generate answers for all evaluation questions using the RAG pipeline.
try:

    generated_answers = []

    retrieved_contexts = []

    for i, question in enumerate(
        test_questions
    ):

        print(
            f"Processing {i+1}/{len(test_questions)}"
        )

        result = (
            ask_question(
                question
            )
        )

        generated_answers.append(
            result["answer"]
        )

        retrieved_contexts.append(
            result["contexts"]
        )

    print(
        "Generation completed!"
    )

except Exception as e:

    print(
        "Generation Error:"
    )

    print(e)

Processing 1/10
Processing 2/10
Processing 3/10
Processing 4/10
Processing 5/10
Processing 6/10
Processing 7/10
Processing 8/10
Processing 9/10
Processing 10/10
Generation completed!


In [11]:
# Create Evaluation Dataset.
#Create a RAGAS-compatible dataset using generated answers, retrieved contexts, and ground truth responses.


from datasets import Dataset
try:
    evaluation_dataset = (
        Dataset.from_dict(
            {
                "question":
                test_questions,

                "answer":
                generated_answers,

                "contexts":
                retrieved_contexts,

                "ground_truth":
                ground_truths
            }
        )
    )

    print(
        evaluation_dataset
    )

except Exception as e:

    print(
        "Dataset Error:"
    )

    print(e)

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 10
})


In [12]:
from langchain_groq import ChatGroq
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings
)

try:
    evaluator_llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=userdata.get(
            "GROQ_KEY"
        )
    )

    evaluator_embeddings = (
        GoogleGenerativeAIEmbeddings(
            model="models/gemini-embedding-001",
            google_api_key=userdata.get(
                "GEMINI_KEY"
            )
        )
    )

    print(
        "Evaluator configured successfully!"
    )

except Exception as e:
    print(
        "Configuration Error:"
    )
    print(e)

Evaluator configured successfully!


In [13]:
# Import RAGAS Metrics
from ragas import evaluate

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

try:
    print(
        "Metrics imported successfully!"
    )

except Exception as e:
    print(
        "Import Error:"
    )
    print(e)

Metrics imported successfully!


/tmp/ipykernel_1888/2433796903.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_1888/2433796903.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_1888/2433796903.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_1888/2433796903.py:4: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed

In [14]:
try:

    results = evaluate(
    evaluation_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

    print(
        "Evaluation completed successfully!"
    )

    print(results)

except Exception as e:

    print(
        "Evaluation Error:"
    )

    print(e)

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[5]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
ERROR:ragas.executor:Exception raised in Job[3]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[4]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[11]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[15]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[2]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[8]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[12]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[16]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[19]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[20]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[21]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : num

Evaluation completed successfully!
{'faithfulness': 1.0000, 'answer_relevancy': 0.8007, 'context_precision': 0.9722, 'context_recall': 0.7500}


In [15]:
try:

    results_df = results.to_pandas()

    print(results_df)
    results_df.to_csv(
        "ragas_results.csv",
        index=False
    )

    print(
        "Results saved successfully!"
    )

except Exception as e:

    print(e)

                                          user_input  \
0                          What is Machine Learning?   
1       What are the main types of Machine Learning?   
2                          What is a Neural Network?   
3                    How does a Neural Network work?   
4                             What is Deep Learning?   
5  What is the difference between Machine Learnin...   
6                       What is a Transformer Model?   
7      What is self-attention in Transformer models?   
8               What is Natural Language Processing?   
9  What are the applications of Natural Language ...   

                                  retrieved_contexts  \
0  [Machine learning (ML) is a field of study in ...   
1  [Approaches\nMachine learning approaches are t...   
2  [A neural network is a group of interconnected...   
3  [In machine learning\nIn machine learning, a n...   
4  [In machine learning, deep learning (DL) focus...   
5  [In machine learning, deep learning (DL) foc